In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q optuna torch joblib pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 32.0 MB/s eta 0:00:00


# 02b — Training Replicas (MLP-PLR)

Trains R MLP-PLR replicas per window pair (A, B) on **Colab GPU**, following the same
structure and output contract as notebook 02. MLP-PLR uses **Periodic → Linear → ReLU**
embeddings per numeric feature (Gorishniy et al. 2022, *On Embeddings for Numerical
Features in Tabular Deep Learning*) followed by an MLP backbone on the concatenated
embedded numeric + raw binary features.

The output schema matches notebook 02 **exactly**, so notebooks 03, 03b and 04 consume
MLP-PLR results the same way they consume XGBoost/LR results.

## Inputs / Outputs

**Input:** `data/processed/`, `data/windows/window_config.json`
**Output per pair:** `data/models/mlp_plr/pair_{pid:02d}/`
- `replicas_A/model_r{r}.joblib`, `replicas_B/model_r{r}.joblib` — per-replica joblib bundles `{state_dict (CPU), arch_config (plain dict), scaler (StandardScaler)}`
- `replicas_{A,B}/training_log_r{r}.csv` — per-epoch train loss & val PR-AUC
- `replicas_{A,B}/seeds_r{r}.json` — exact bootstrap/model seeds
- `hparams_A.json`, `hparams_B.json` — Optuna-tuned hyperparameters per window
- `reference_scaler.joblib` — StandardScaler fit on window A's numeric columns (for notebook 04's covariate-drift calculation)
- `predictions.npz` — 12-key schema identical to notebook 02

## Runtime (T4 GPU rough estimate)

- Optuna tuning: `N_TRIALS_MLP` × `CV_N_SPLITS` folds per window ≈ 30–60 min/window
- Final replica training: R × ~1 min each ≈ 10–20 min/window
- **Total: ~2–3 h per pair.** Reduce `N_TRIALS_MLP` or `CV_N_SPLITS` to shrink.

## GPU reproducibility caveat

Training on GPU is not bit-reproducible even with fixed seeds — cuDNN's non-deterministic
algorithms introduce small variance that persists under `cudnn.deterministic=True`.


In [3]:
import json
import math
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

optuna.logging.set_verbosity(optuna.logging.WARNING)

assert torch.cuda.is_available(), 'MLP-PLR training requires a GPU runtime.'
DEVICE = torch.device('cuda')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {torch.cuda.get_device_name(0)}')

WORKSPACE = Path('/content/drive/MyDrive/Homesite_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Fixed training knobs (not tuned) ──────────────────────────────────────
MLP_FIXED = dict(
    max_epochs        = 50,    # cap for final replica training
    tuning_max_epochs = 20,    # shorter cap during Optuna trials
    patience          = 10,    # early-stop patience on val PR-AUC
    batch_size        = 1024,
    val_tail_frac     = 0.15,  # fraction of the shuffled bootstrap held out for ES
)

# ── Tuning configuration ──────────────────────────────────────────────────
N_TRIALS_MLP = 20    # Optuna trials per window
CV_N_SPLITS  = 3     # TimeSeriesSplit folds (matches notebook 02)

print('Imports OK')

PyTorch : 2.10.0+cu128
Device  : NVIDIA L4
Imports OK


In [4]:
# ── Model type — hardcoded for this notebook ──────────────────────────────
MODEL_TYPE = 'mlp_plr'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODEL_TYPE : {MODEL_TYPE}')
print(f'MODEL_DIR  : {MODEL_DIR}')

MODEL_TYPE : mlp_plr
MODEL_DIR  : /content/drive/MyDrive/Homesite_workspace/data/models/mlp_plr


## 1. Load processed data and window config

In [5]:
X_df = pd.read_parquet(PROC_DIR / 'X.parquet')
X    = X_df.values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R           = config['parameters']['R']
K_FRAC      = config['parameters']['K_FRAC']
PAIR_STRIDE = config['parameters']['PAIR_STRIDE']
pairs       = config['pairs']

all_cols    = feature_names_json['all']
num_col_idx = np.array([all_cols.index(c) for c in feature_names_json['num']], dtype=np.int64)
bin_col_idx = np.array([all_cols.index(c) for c in feature_names_json['bin']], dtype=np.int64)

print(f'X: {X.shape}, Y: {Y.shape}')
print(f'R={R}, K_FRAC={K_FRAC}, PAIR_STRIDE={PAIR_STRIDE}, {len(pairs)} pairs')
print(f'Numeric: {len(num_col_idx)}   Binary: {len(bin_col_idx)}   Total: {X.shape[1]}')

X: (260753, 317), Y: (260753,)
R=3, K_FRAC=0.1, PAIR_STRIDE=4, 5 pairs
Numeric: 280   Binary: 37   Total: 317


## 2. MLP-PLR architecture

Per Gorishniy et al. 2022:
- **PLREmbedding** — for each numeric feature j, sample k frequencies `f_j ~ N(0, σ²)`,
  compute `[sin(2π f_j x_j), cos(2π f_j x_j)]` (Periodic), feed through a per-feature
  Linear (Linear), then ReLU. Output dim per feature = `d_emb`.
- **MLPPLR** — concatenates `(p_num · d_emb)` embedded numerics with raw binaries, runs
  through `n_blocks` Linear+ReLU+Dropout blocks, ends with a single linear head.

The whole architecture is small enough to keep state_dicts on disk via joblib.

In [6]:
class PLREmbedding(nn.Module):
    """Periodic-Linear-ReLU embedding for numeric features.

    For each of the p_num scalar inputs, produces a vector of dimension d_emb.
    Output shape: (batch, p_num, d_emb), flattened to (batch, p_num * d_emb)
    by the caller.
    """
    def __init__(self, n_features: int, k: int, d_emb: int, sigma_init: float):
        super().__init__()
        self.n_features = n_features
        self.k          = k
        self.d_emb      = d_emb

        # Per-feature Fourier frequencies, shape (n_features, k). Initialised
        # from N(0, sigma_init²) and TRAINABLE (becomes a hyperparameter that
        # the model learns to refine during training).
        self.freqs = nn.Parameter(torch.randn(n_features, k) * sigma_init)

        # Per-feature linear: maps (2k,) -> (d_emb,). Implemented as a single
        # Linear with grouped weights would be more efficient; we use a tiny
        # explicit per-feature linear via a 3-D weight tensor.
        self.weight = nn.Parameter(torch.empty(n_features, 2 * k, d_emb))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_emb))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x_num: torch.Tensor) -> torch.Tensor:
        # x_num: (batch, n_features)
        # Periodic step
        scaled = 2 * math.pi * x_num.unsqueeze(-1) * self.freqs   # (batch, n_features, k)
        periodic = torch.cat([torch.sin(scaled), torch.cos(scaled)], dim=-1)  # (b, p, 2k)
        # Per-feature Linear via einsum
        out = torch.einsum('bpd,pde->bpe', periodic, self.weight) + self.bias
        out = F.relu(out)
        return out.flatten(start_dim=1)   # (batch, n_features * d_emb)


class MLPPLR(nn.Module):
    """PLR embedding for numerics + raw binaries → MLP backbone → 1-logit head."""
    def __init__(self, n_num: int, n_bin: int,
                 k: int, d_emb: int, sigma_init: float,
                 n_blocks: int, d_block: int, dropout: float):
        super().__init__()
        self.n_num = n_num
        self.n_bin = n_bin

        if n_num > 0:
            self.embedding = PLREmbedding(n_num, k, d_emb, sigma_init)
            input_dim = n_num * d_emb + n_bin
        else:
            self.embedding = None
            input_dim = n_bin

        layers = []
        d_in = input_dim
        for _ in range(n_blocks):
            layers.append(nn.Linear(d_in, d_block))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            d_in = d_block
        layers.append(nn.Linear(d_in, 1))
        self.backbone = nn.Sequential(*layers)

    def forward(self, x_num: torch.Tensor, x_bin: torch.Tensor) -> torch.Tensor:
        if self.embedding is not None:
            emb = self.embedding(x_num)
            h = torch.cat([emb, x_bin], dim=1) if self.n_bin > 0 else emb
        else:
            h = x_bin
        return self.backbone(h).squeeze(-1)


def build_arch_config(trial_or_dict, n_num: int, n_bin: int) -> dict:
    """Standard hyperparameter dict shape used everywhere in this notebook."""
    g = (trial_or_dict.suggest_int     if hasattr(trial_or_dict, 'suggest_int')
         else lambda name, *a, **kw: trial_or_dict[name])
    gf = (trial_or_dict.suggest_float  if hasattr(trial_or_dict, 'suggest_float')
         else lambda name, *a, **kw: trial_or_dict[name])
    return {
        'n_num':      n_num,
        'n_bin':      n_bin,
        'k':          g('k',          4, 32),
        'd_emb':      g('d_emb',      8, 32),
        'sigma_init': gf('sigma_init', 1e-2, 1.0, log=True),
        'n_blocks':   g('n_blocks',   1, 4),
        'd_block':    g('d_block',    64, 512, log=True),
        'dropout':    gf('dropout',   0.0, 0.5),
    }


print('Architecture defined.')

Architecture defined.


## 3. Helper functions

Same conventions as notebook 02 — stratified bootstrap, top-K flagging — plus a per-replica training loop with chronological-tail validation for early stopping.

In [7]:
SEED_BASE = 42

def stratified_bootstrap(idx: np.ndarray, Y: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    Y_sub = Y[idx]
    pos = idx[Y_sub == 1]
    neg = idx[Y_sub == 0]
    boot_pos = rng.choice(pos, size=len(pos), replace=True)
    boot_neg = rng.choice(neg, size=len(neg), replace=True)
    return np.concatenate([boot_pos, boot_neg])


def compute_flagged_topk(p_hat_A: np.ndarray, p_hat_B: np.ndarray, k_frac: float) -> np.ndarray:
    score = np.maximum(p_hat_A, p_hat_B)
    n_flag = max(int(np.ceil(len(score) * k_frac)), 1)
    return np.argsort(-score)[:n_flag].astype(np.int64)


def scale_numeric_inplace(X_raw: np.ndarray, scaler: StandardScaler, num_idx: np.ndarray) -> np.ndarray:
    """Return a copy of X_raw with numeric columns standardised; binaries untouched."""
    out = X_raw.astype(np.float32, copy=True)
    out[:, num_idx] = scaler.transform(X_raw[:, num_idx]).astype(np.float32)
    return out


def split_num_bin(X_scaled: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Slice the (already scaled) feature matrix into the (num, bin) tensors used by the model."""
    return X_scaled[:, num_col_idx], X_scaled[:, bin_col_idx]

In [8]:
def train_one_replica(arch_config: dict,
                      X_train_num: np.ndarray, X_train_bin: np.ndarray, Y_train: np.ndarray,
                      X_val_num:   np.ndarray, X_val_bin:   np.ndarray, Y_val:   np.ndarray,
                      max_epochs: int, patience: int, batch_size: int,
                      lr: float, weight_decay: float, model_seed: int):
    """Train one MLP-PLR; return (model, training_log_df, best_val_pr_auc).

    Early stopping on validation PR-AUC. Best-state restoration at the end.
    """
    torch.manual_seed(model_seed)
    torch.cuda.manual_seed_all(model_seed)

    model = MLPPLR(**arch_config).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    X_tr_n = torch.from_numpy(X_train_num).float().to(DEVICE)
    X_tr_b = torch.from_numpy(X_train_bin).float().to(DEVICE)
    Y_tr_t = torch.from_numpy(Y_train.astype(np.float32)).to(DEVICE)
    X_va_n = torch.from_numpy(X_val_num).float().to(DEVICE)
    X_va_b = torch.from_numpy(X_val_bin).float().to(DEVICE)
    Y_va_np = Y_val.astype(np.float32)

    n = len(Y_train)
    rng = np.random.default_rng(model_seed)

    best_pr   = -np.inf
    best_state = None
    bad_epochs = 0
    log_rows  = []

    for epoch in range(max_epochs):
        # Shuffle each epoch
        perm = rng.permutation(n)
        model.train()
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            sel = perm[start:start + batch_size]
            optim.zero_grad()
            logits = model(X_tr_n[sel], X_tr_b[sel])
            loss = loss_fn(logits, Y_tr_t[sel])
            loss.backward()
            optim.step()
            epoch_loss += float(loss.item()) * len(sel)
        epoch_loss /= n

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va_n, X_va_b)
            val_probs  = torch.sigmoid(val_logits).cpu().numpy()
        val_pr = float(average_precision_score(Y_va_np, val_probs))

        log_rows.append({'epoch': epoch, 'train_loss': epoch_loss, 'val_pr_auc': val_pr})

        if val_pr > best_pr:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(log_rows), best_pr

In [9]:
def tune_hyperparameters_mlp(window_idx: np.ndarray, X_all: np.ndarray, Y_all: np.ndarray,
                             study_seed: int, n_trials: int = N_TRIALS_MLP) -> dict:
    """Optuna search over MLP-PLR hyperparameters using TimeSeriesSplit + PR-AUC."""
    X_win = X_all[window_idx]   # already step-sorted
    Y_win = Y_all[window_idx]
    tscv  = TimeSeriesSplit(n_splits=CV_N_SPLITS)

    n_num = len(num_col_idx)
    n_bin = len(bin_col_idx)

    def objective(trial: optuna.Trial) -> float:
        arch    = build_arch_config(trial, n_num, n_bin)
        lr      = trial.suggest_float('lr',           1e-4, 1e-2, log=True)
        wd      = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

        pr_aucs = []
        for fold, (tr, va) in enumerate(tscv.split(X_win)):
            scaler = StandardScaler().fit(X_win[tr][:, num_col_idx])
            X_tr_s = scale_numeric_inplace(X_win[tr], scaler, num_col_idx)
            X_va_s = scale_numeric_inplace(X_win[va], scaler, num_col_idx)

            X_tr_n, X_tr_b = split_num_bin(X_tr_s)
            X_va_n, X_va_b = split_num_bin(X_va_s)

            _, _, best_pr = train_one_replica(
                arch,
                X_tr_n, X_tr_b, Y_win[tr],
                X_va_n, X_va_b, Y_win[va],
                max_epochs   = MLP_FIXED['tuning_max_epochs'],
                patience     = MLP_FIXED['patience'],
                batch_size   = MLP_FIXED['batch_size'],
                lr           = lr,
                weight_decay = wd,
                model_seed   = study_seed + fold,
            )
            pr_aucs.append(best_pr)

        return float(np.mean(pr_aucs))

    sampler = TPESampler(seed=study_seed)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return dict(study.best_trial.params)

## 4. Main training loop

Per pair: tune → bootstrap → train R replicas (with chronological ES tail and per-replica scaler) → evaluate → save. Skips pairs that already have a complete `predictions.npz`.

In [10]:
performance_log = []
n_num = len(num_col_idx)
n_bin = len(bin_col_idx)

for p in pairs:
    pid       = p['pair_id']
    pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
    pred_file = pair_dir / 'predictions.npz'

    if pred_file.exists():
        data = np.load(pred_file)
        if all(k in data.files for k in ('roc_auc_A', 'roc_auc_B')):
            print(f'Pair {pid:02d}: already done, skipping.')
            performance_log.append({
                'pair_id':   pid,
                'pr_auc_A':  float(data['pr_auc_A']),
                'pr_auc_B':  float(data['pr_auc_B']),
                'roc_auc_A': float(data['roc_auc_A']),
                'roc_auc_B': float(data['roc_auc_B']),
                'n_flagged': int(data['flagged_idx'].shape[0]),
            })
            continue
        print(f'Pair {pid:02d}: stale predictions.npz — re-running.')

    print(
        f'\n── Pair {pid:02d}: '
        f'A_end={p["step_label_A"]}  B_end={p["step_label_B"]}  '
        f'eval={p["eval_start_label"]}→{p["eval_end_label"]}  '
        f'|A|={p["n_train_A"]:,} |B|={p["n_train_B"]:,} |eval|={p["n_eval"]:,}  '
        f'PAIR_STRIDE={PAIR_STRIDE} ──'
    )

    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    assert len(set(idx_A.tolist()) & set(idx_eval.tolist())) == 0
    assert len(set(idx_B.tolist()) & set(idx_eval.tolist())) == 0

    pair_dir.mkdir(parents=True, exist_ok=True)
    dir_A = pair_dir / 'replicas_A'
    dir_B = pair_dir / 'replicas_B'
    dir_A.mkdir(exist_ok=True)
    dir_B.mkdir(exist_ok=True)

    # Reference scaler — used by notebook 04 for covariate-drift only.
    ref_scaler = StandardScaler().fit(X[idx_A][:, num_col_idx])
    joblib.dump(ref_scaler, pair_dir / 'reference_scaler.joblib')

    # ── Tune hparams per window ──
    print('  Tuning window A ...')
    hparams_A = tune_hyperparameters_mlp(idx_A, X, Y, study_seed=SEED_BASE + pid * 10 + 1)
    with open(pair_dir / 'hparams_A.json', 'w') as f:
        json.dump(hparams_A, f, indent=2)
    print(f'    best A params: {hparams_A}')

    print('  Tuning window B ...')
    hparams_B = tune_hyperparameters_mlp(idx_B, X, Y, study_seed=SEED_BASE + pid * 10 + 2)
    with open(pair_dir / 'hparams_B.json', 'w') as f:
        json.dump(hparams_B, f, indent=2)
    print(f'    best B params: {hparams_B}')

    # Build the canonical arch_config (a plain dict, not Optuna trial) per window
    arch_A = build_arch_config(hparams_A, n_num, n_bin)
    arch_B = build_arch_config(hparams_B, n_num, n_bin)

    X_eval = X[idx_eval]
    Y_eval = Y[idx_eval]

    # ── Chronological ES validation split (held out BEFORE bootstrapping) ──
    val_frac = MLP_FIXED['val_tail_frac']

    n_es_A     = max(int(np.ceil(len(idx_A) * val_frac)), 1)
    idx_A_boot = idx_A[:-n_es_A]
    X_va_A_raw = X[idx_A[-n_es_A:]]
    Y_va_A     = Y[idx_A[-n_es_A:]].astype(np.float32)

    n_es_B     = max(int(np.ceil(len(idx_B) * val_frac)), 1)
    idx_B_boot = idx_B[:-n_es_B]
    X_va_B_raw = X[idx_B[-n_es_B:]]
    Y_va_B     = Y[idx_B[-n_es_B:]].astype(np.float32)

    # ── Train R replicas for window A ──
    preds_A          = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_A = np.zeros(R, dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + r * 2
        model_seed = SEED_BASE + pid * 10_000 + r * 2 + 1
        boot_idx   = stratified_bootstrap(idx_A_boot, Y, seed=boot_seed)

        # Per-replica scaler (fit on the bootstrap sample)
        rep_scaler = StandardScaler().fit(X[boot_idx][:, num_col_idx])
        X_tr_s     = scale_numeric_inplace(X[boot_idx], rep_scaler, num_col_idx)
        X_va_s     = scale_numeric_inplace(X_va_A_raw,  rep_scaler, num_col_idx)
        X_ev_s     = scale_numeric_inplace(X_eval,      rep_scaler, num_col_idx)

        X_tr_n, X_tr_b = split_num_bin(X_tr_s)
        X_va_n, X_va_b = split_num_bin(X_va_s)
        X_ev_n, X_ev_b = split_num_bin(X_ev_s)

        model, log, _ = train_one_replica(
            arch_A,
            X_tr_n, X_tr_b, Y[boot_idx],
            X_va_n, X_va_b, Y_va_A,
            max_epochs   = MLP_FIXED['max_epochs'],
            patience     = MLP_FIXED['patience'],
            batch_size   = MLP_FIXED['batch_size'],
            lr           = hparams_A.get('lr', 1e-3),
            weight_decay = hparams_A.get('weight_decay', 1e-4),
            model_seed   = model_seed,
        )

        # Predict on eval
        model.eval()
        with torch.no_grad():
            ev_n_t = torch.from_numpy(X_ev_n).float().to(DEVICE)
            ev_b_t = torch.from_numpy(X_ev_b).float().to(DEVICE)
            preds_A[r] = torch.sigmoid(model(ev_n_t, ev_b_t)).cpu().numpy()

        per_rep_pr_auc_A[r] = average_precision_score(Y_eval, preds_A[r])

        # Save bundle: state_dict (CPU) + arch_config (plain dict) + scaler
        bundle = {
            'state_dict':  {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'arch_config': arch_A,
            'scaler':      rep_scaler,
        }
        joblib.dump(bundle, dir_A / f'model_r{r}.joblib', compress=3)
        log.to_csv(dir_A / f'training_log_r{r}.csv', index=False)
        with open(dir_A / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)
        print(f'  A replica {r}: PR-AUC = {per_rep_pr_auc_A[r]:.4f}')

        del model, ev_n_t, ev_b_t
        torch.cuda.empty_cache()

    # ── Train R replicas for window B ──
    preds_B          = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_B = np.zeros(R, dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + 5_000 + r * 2
        model_seed = SEED_BASE + pid * 10_000 + 5_000 + r * 2 + 1
        boot_idx   = stratified_bootstrap(idx_B_boot, Y, seed=boot_seed)

        rep_scaler = StandardScaler().fit(X[boot_idx][:, num_col_idx])
        X_tr_s     = scale_numeric_inplace(X[boot_idx], rep_scaler, num_col_idx)
        X_va_s     = scale_numeric_inplace(X_va_B_raw,  rep_scaler, num_col_idx)
        X_ev_s     = scale_numeric_inplace(X_eval,      rep_scaler, num_col_idx)

        X_tr_n, X_tr_b = split_num_bin(X_tr_s)
        X_va_n, X_va_b = split_num_bin(X_va_s)
        X_ev_n, X_ev_b = split_num_bin(X_ev_s)

        model, log, _ = train_one_replica(
            arch_B,
            X_tr_n, X_tr_b, Y[boot_idx],
            X_va_n, X_va_b, Y_va_B,
            max_epochs   = MLP_FIXED['max_epochs'],
            patience     = MLP_FIXED['patience'],
            batch_size   = MLP_FIXED['batch_size'],
            lr           = hparams_B.get('lr', 1e-3),
            weight_decay = hparams_B.get('weight_decay', 1e-4),
            model_seed   = model_seed,
        )

        model.eval()
        with torch.no_grad():
            ev_n_t = torch.from_numpy(X_ev_n).float().to(DEVICE)
            ev_b_t = torch.from_numpy(X_ev_b).float().to(DEVICE)
            preds_B[r] = torch.sigmoid(model(ev_n_t, ev_b_t)).cpu().numpy()

        per_rep_pr_auc_B[r] = average_precision_score(Y_eval, preds_B[r])

        bundle = {
            'state_dict':  {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'arch_config': arch_B,
            'scaler':      rep_scaler,
        }
        joblib.dump(bundle, dir_B / f'model_r{r}.joblib', compress=3)
        log.to_csv(dir_B / f'training_log_r{r}.csv', index=False)
        with open(dir_B / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)
        print(f'  B replica {r}: PR-AUC = {per_rep_pr_auc_B[r]:.4f}')

        del model, ev_n_t, ev_b_t
        torch.cuda.empty_cache()

    # ── Aggregate, flag, score ──
    p_hat_A = preds_A.mean(axis=0)
    p_hat_B = preds_B.mean(axis=0)
    flagged_local_idx = compute_flagged_topk(p_hat_A, p_hat_B, K_FRAC)

    pr_auc_A  = average_precision_score(Y_eval, p_hat_A)
    pr_auc_B  = average_precision_score(Y_eval, p_hat_B)
    roc_auc_A = roc_auc_score(Y_eval, p_hat_A)
    roc_auc_B = roc_auc_score(Y_eval, p_hat_B)

    print(f'  Averaged: PR-AUC A = {pr_auc_A:.4f}, PR-AUC B = {pr_auc_B:.4f}')
    print(f'            ROC-AUC A = {roc_auc_A:.4f}, ROC-AUC B = {roc_auc_B:.4f}')
    print(f'  Flagged (top {K_FRAC:.0%}): {flagged_local_idx.shape[0]:,} / {len(idx_eval):,}')

    np.savez_compressed(
        pred_file,
        preds_A              = preds_A,
        preds_B              = preds_B,
        p_hat_A              = p_hat_A,
        p_hat_B              = p_hat_B,
        flagged_idx          = flagged_local_idx,
        Y_eval               = Y_eval,
        pr_auc_A             = np.float32(pr_auc_A),
        pr_auc_B             = np.float32(pr_auc_B),
        roc_auc_A            = np.float32(roc_auc_A),
        roc_auc_B            = np.float32(roc_auc_B),
        per_replica_pr_auc_A = per_rep_pr_auc_A,
        per_replica_pr_auc_B = per_rep_pr_auc_B,
    )

    performance_log.append({
        'pair_id':   pid,
        'pr_auc_A':  pr_auc_A,
        'pr_auc_B':  pr_auc_B,
        'roc_auc_A': roc_auc_A,
        'roc_auc_B': roc_auc_B,
        'n_flagged': flagged_local_idx.shape[0],
    })

print('\n✓ All window pairs processed.')


── Pair 00: A_end=2013-08  B_end=2013-10  eval=2013-11→2014-01  |A|=71,989 |B|=78,443 |eval|=23,346  PAIR_STRIDE=4 ──
  Tuning window A ...
    best A params: {'k': 32, 'd_emb': 10, 'sigma_init': 0.01141204913142774, 'n_blocks': 2, 'd_block': 311, 'dropout': 0.359376646770003, 'lr': 0.000599870905448968, 'weight_decay': 0.0013116281298813693}
  Tuning window B ...
    best B params: {'k': 24, 'd_emb': 27, 'sigma_init': 0.02006308521239954, 'n_blocks': 2, 'd_block': 164, 'dropout': 0.24032124868013344, 'lr': 0.001579295495292571, 'weight_decay': 4.003052830991666e-05}
  A replica 0: PR-AUC = 0.8721
  A replica 1: PR-AUC = 0.8715
  A replica 2: PR-AUC = 0.8709
  B replica 0: PR-AUC = 0.8770
  B replica 1: PR-AUC = 0.8727
  B replica 2: PR-AUC = 0.8706
  Averaged: PR-AUC A = 0.8780, PR-AUC B = 0.8812
            ROC-AUC A = 0.9569, ROC-AUC B = 0.9581
  Flagged (top 10%): 2,335 / 23,346

── Pair 01: A_end=2013-12  B_end=2014-02  eval=2014-03→2014-05  |A|=74,193 |B|=76,161 |eval|=30,717  P

## 5. Save and summarize performance

In [11]:
perf_df = pd.DataFrame(performance_log)
perf_df.to_csv(MODEL_DIR / 'performance_log.csv', index=False)

print(f'Saved {MODEL_DIR / "performance_log.csv"}')
print()
print(perf_df.round(4).to_string(index=False))

Saved /content/drive/MyDrive/Homesite_workspace/data/models/mlp_plr/performance_log.csv

 pair_id  pr_auc_A  pr_auc_B  roc_auc_A  roc_auc_B  n_flagged
       0    0.8780    0.8812     0.9569     0.9581       2335
       1    0.8930    0.8922     0.9641     0.9642       3072
       2    0.8696    0.8736     0.9588     0.9591       2589
       3    0.8340    0.8338     0.9406     0.9429       2414
       4    0.8686    0.8725     0.9533     0.9553       2854
